In [5]:
import json

class ReflectionAgent:
    def __init__(self, file):
        self.nodes = {}
        self.state = {"answers": {}, "axis1": [], "axis2": [], "axis3": []}
        self.load_tree(file)

    def load_tree(self, file):
        with open(file, 'r', encoding='utf-8') as f:
            data = json.load(f)

        for node in data['nodes']:
            self.nodes[node['id']] = node

    def run(self):
        current = "START"

        while current:
            node = self.nodes[current]
            ntype = node['type']

            if ntype == "start":
                print(node['text'])
                current = self.get_next(current)

            elif ntype == "question":
                print("\n" + node['text'])
                options = node.get('options', [])

                for i, opt in enumerate(options, 1):
                    print(f"{i}. {opt}")

                choice = int(input("Choose: ")) - 1
                answer = options[choice]
                self.state["answers"][current] = answer

                if node.get('signal'):
                    axis, val = node['signal'].split(":")
                    self.state[axis].append(val)

                current = self.get_next(current)

            elif ntype == "decision":
                prev = self.get_parent(current)
                prev_answer = self.state["answers"].get(prev)

                rules = node.get('rules', [])
                for rule in rules:
                    if prev_answer in rule['answers']:
                        current = rule['target']
                        break

            elif ntype == "reflection":
                print("\n" + node['text'])

                if node.get('signal'):
                    axis, val = node['signal'].split(":")
                    self.state[axis].append(val)

                input("Press Enter to continue...")
                current = self.get_next(current)

            elif ntype == "bridge":
                print("\n" + node['text'])
                current = node.get('target')

            elif ntype == "summary":
                print("\n--- SUMMARY ---")
                print(node['text'].format(
                    axis1=self.get_dominant("axis1"),
                    axis2=self.get_dominant("axis2"),
                    axis3=self.get_dominant("axis3")
                ))
                current = self.get_next(current)

            elif ntype == "end":
                print("\n" + node['text'])
                break

    def get_next(self, current):
        for node_id, node in self.nodes.items():
            if node.get('parentId') == current:
                return node_id
        return None

    def get_parent(self, node_id):
        return self.nodes[node_id].get('parentId')

    def get_dominant(self, axis):
        if not self.state[axis]:
            return "neutral"
        return max(set(self.state[axis]), key=self.state[axis].count)


if __name__ == "__main__":
    agent = ReflectionAgent("reflection-tree.json")
    agent.run()

Good evening. Let's reflect on your day.

How would you describe your day?
1. Productive
2. Tough
3. Mixed
4. Frustrating
Choose: 3

When things went well, why?
1. My effort
2. Team support
3. Luck
4. Adaptability
Choose: 4

You showed agency today — your actions influenced outcomes.
Press Enter to continue...

Now let's think about your contribution today.

What best describes your actions today?
1. Helped others
2. Did my work
3. Expected recognition
4. Felt unsupported
Choose: 2

You focused on contributing — this builds long-term value.
Press Enter to continue...

Now let's expand perspective beyond yourself.

Who were you thinking about most today?
1. Myself
2. Team
3. Colleague
4. Customer
Choose: 1

Your focus stayed on yourself — consider how others were impacted.
Press Enter to continue...
